# Protein Data

> Protein domain retrieval and genomic coordinate mapping

In [ ]:
#| default_exp protein_data

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import requests
import numpy as np
from functools import lru_cache
from typing import List, Optional, Tuple, Dict

try:
    from allos.transcript_data import TranscriptData
except ImportError:
    # For notebook development before export
    TranscriptData = None

## Helper Functions

Private helper functions for API calls and data processing.

In [ ]:
#| export
def _strip_version(tid: str) -> str:
    """Drop .version suffix from Ensembl IDs if present."""
    if "." in tid:
        base, _, tail = tid.rpartition(".")
        if tail.isdigit():
            return base
    return tid

### Ensembl REST API

In [ ]:
#| export
ENSEMBL = "https://rest.ensembl.org"
HEADERS_ENSEMBL = {
    "Content-Type": "application/json",
    "User-Agent": "AllosProteinData/1.0",
}

def _rest_ensembl(url: str):
    """REST wrapper for Ensembl API."""
    r = requests.get(url, headers=HEADERS_ENSEMBL, timeout=60)
    r.raise_for_status()
    return r.json()


@lru_cache(maxsize=4096)
def _ensembl_lookup_expand(tid_clean: str):
    """Lookup transcript with expand=1 to get Translation info."""
    return _rest_ensembl(f"{ENSEMBL}/lookup/id/{tid_clean}?expand=1")


@lru_cache(maxsize=4096)
def _ensembl_xrefs_uniprot(ens_protein_id: str):
    """
    Map ENSP -> UniProt accessions via Ensembl xrefs.
    Returns list of accessions (strings), best-first.
    """
    # Try SwissProt first, then TrEMBL
    accs = []
    for db in ("UniProt/SWISSPROT", "UniProt/SPTREMBL"):
        try:
            js = _rest_ensembl(f"{ENSEMBL}/xrefs/id/{ens_protein_id}?external_db={db}")
            for hit in js or []:
                pid = hit.get("primary_id") or hit.get("display_id") or hit.get("id")
                if pid:
                    accs.append(str(pid))
        except Exception:
            pass

    # de-dup preserving order
    seen, out = set(), []
    for a in accs:
        if a not in seen:
            out.append(a)
            seen.add(a)
    return out

In [ ]:
#| export
@lru_cache(maxsize=4096)
def _fetch_ensembl_translation_and_features(tid: str):
    """
    Ensembl provider:
      transcript -> ENSP + protein length + list of features from overlap/translation
    """
    tid_clean = _strip_version(tid)
    js = _ensembl_lookup_expand(tid_clean)
    tr = js.get("Translation")
    if not tr:
        return None, 0, []

    pid = tr["id"]                 # ENSP...
    plen = int(tr["length"])

    feats_raw = []
    _n_errors = 0
    _last_err = None
    for ft in ("domain", "family", "protein_feature"):
        try:
            feats_raw += _rest_ensembl(f"{ENSEMBL}/overlap/translation/{pid}?feature={ft}")
        except Exception as e:
            _n_errors += 1
            _last_err = e

    # If every feature call failed (e.g. timeout), raise so lru_cache does not
    # store the empty result — the next call will retry the API.
    if _n_errors == 3:
        raise RuntimeError(f"All Ensembl feature lookups failed for {pid}") from _last_err

    direct, sifts = [], []
    for f in feats_raw:
        desc = f.get("description") or ""
        if f.get("feature_type") == "supporting_region":
            continue

        dom_id = (
            f.get("id")
            or f.get("external_name")
            or f.get("feature_type", "feat")
        )
        source = (
            f.get("logic_name")
            or (f.get("analysis") or {}).get("logic_name")
            or f.get("feature_type")
            or "unknown"
        )

        try:
            entry = {
                "start": int(f["start"]),
                "end": int(f["end"]),
                "id": str(dom_id),
                "desc": str(desc),
                "source": str(source),
            }
        except Exception:
            continue

        if desc.startswith("Via SIFTS"):
            sifts.append(entry)
        else:
            direct.append(entry)

    # Prefer direct functional annotations; fall back to SIFTS structural
    # mappings when a transcript has no direct domain predictions.
    out = direct if direct else sifts

    # de-dup
    uniq, seen = [], set()
    for d in sorted(out, key=lambda x: (x["start"], x["end"], x["id"], x["source"])):
        key = (d["start"], d["end"], d["id"], d["source"])
        if key not in seen:
            uniq.append(d)
            seen.add(key)

    return pid, plen, uniq

### InterPro REST API

In [ ]:
#| export
INTERPRO = "https://www.ebi.ac.uk/interpro/api"
HEADERS_INTERPRO = {
    "Accept": "application/json",
    "User-Agent": "AllosProteinData/1.0",
}

def _rest_interpro(url: str):
    """REST wrapper for InterPro API."""
    r = requests.get(url, headers=HEADERS_INTERPRO, timeout=30)
    r.raise_for_status()
    return r.json()

def _iter_interpro_pages(url: str, max_pages: int = 50):
    """
    InterPro API is paginated and typically returns a dict with 'results' and 'next'.
    This generator yields each result object.
    """
    pages = 0
    while url and pages < max_pages:
        js = _rest_interpro(url)
        results = js.get("results", js if isinstance(js, list) else [])
        if isinstance(results, list):
            for item in results:
                yield item
        url = js.get("next", None) if isinstance(js, dict) else None
        pages += 1


@lru_cache(maxsize=4096)
def _fetch_interpro_features_from_uniprot(uniprot_acc: str):
    """
    InterPro provider:
      UniProt accession -> InterPro entries with protein locations -> fragments (AA coords)
    Returns list of feature dicts in the SAME format as Ensembl provider:
        {"start","end","id","desc","source"}
    """
    # This is the commonly supported InterPro API route for protein matches:
    # entry/all/protein/uniprot/{acc}/
    url = f"{INTERPRO}/entry/all/protein/uniprot/{uniprot_acc}/?page_size=200"

    out = []
    for result in _iter_interpro_pages(url):
        meta = result.get("metadata") or {}
        proteins = result.get("proteins") or []

        source_db = meta.get("source_database") or meta.get("database") or meta.get("source") or "interpro"
        acc = meta.get("accession") or meta.get("id") or meta.get("identifier")
        name = meta.get("name") or meta.get("description") or acc or "feature"

        if not proteins:
            continue

        for p in proteins:
            locs = p.get("entry_protein_locations") or p.get("protein_locations") or p.get("locations") or []
            for loc in locs:
                frags = loc.get("fragments") or []
                for frag in frags:
                    try:
                        start = int(frag.get("start"))
                        end = int(frag.get("end"))
                    except Exception:
                        continue
                    if start <= 0 or end <= 0 or end < start:
                        continue

                    out.append(
                        {
                            "start": start,
                            "end": end,
                            "id": str(acc or name),
                            "desc": str(name),
                            "source": str(source_db),
                        }
                    )

    # de-dup
    uniq, seen = [], set()
    for d in sorted(out, key=lambda x: (x["start"], x["end"], x["id"], x["source"])):
        key = (d["start"], d["end"], d["id"], d["source"])
        if key not in seen:
            uniq.append(d)
            seen.add(key)
    return uniq

### Unified Translation & Features Fetcher

In [ ]:
#| export
def _fetch_translation_and_features(tid: str, provider: str = "ensembl"):
    """
    Unified entrypoint.

    provider="ensembl":
        Uses Ensembl overlap/translation endpoints (fast, ENSP-native).
    provider="interpro":
        Maps ENSP -> UniProt, then uses InterPro API to fetch matches.
        (Still returns AA coords, so downstream mapping works unchanged.)
    """
    provider = (provider or "ensembl").lower()

    # Always start from Ensembl lookup to get translation + length
    tid_clean = _strip_version(tid)
    js = _ensembl_lookup_expand(tid_clean)
    tr = js.get("Translation")
    if not tr:
        return None, 0, []

    ens_pid = tr["id"]
    plen = int(tr["length"])

    if provider == "ensembl":
        pid, plen2, feats = _fetch_ensembl_translation_and_features(tid)
        return pid, plen2, feats

    if provider == "interpro":
        uniprot_accs = _ensembl_xrefs_uniprot(ens_pid)
        if not uniprot_accs:
            # no UniProt mapping => cannot query InterPro protein endpoint
            return ens_pid, plen, []

        # Use the first accession as "best" (SwissProt preferred by _ensembl_xrefs_uniprot)
        feats = _fetch_interpro_features_from_uniprot(uniprot_accs[0])
        return ens_pid, plen, feats

    raise ValueError(f"Unknown provider={provider!r}. Use 'ensembl' or 'interpro'.")

### Feature Processing

In [ ]:
#| export
def _features_to_domains_aa(
    features,
    *,
    max_domains: int | None = None,
    id_prefixes: list[str] | None = None,
):
    """
    Collapse protein_features into AA-domain dicts:
      [{'name': <stable_id>, 'label': <desc>, 'start': aa_start, 'end': aa_end}, ...]
    """
    if id_prefixes is not None:
        features = [
            f for f in features
            if any(str(f.get("id", "")).startswith(p) for p in id_prefixes)
        ]

    by_id: dict[str, list[dict]] = {}
    for f in features:
        dom_id = str(f.get("id", "") or "")
        if not dom_id:
            continue
        by_id.setdefault(dom_id, []).append(f)

    domains = []
    for dom_id, feats in by_id.items():
        aa_start = min(f["start"] for f in feats)
        aa_end = max(f["end"] for f in feats)

        descs = [(f.get("desc") or "").strip() for f in feats]
        descs = [d for d in descs if d]
        desc = max(descs, key=len) if descs else dom_id

        label = desc if len(desc) <= 60 else (desc[:57] + "…")
        domains.append({"name": dom_id, "label": label, "start": aa_start, "end": aa_end})

    domains.sort(key=lambda d: d["end"] - d["start"], reverse=True)
    if max_domains is not None:
        domains = domains[:max_domains]
    return domains

### Genomic Coordinate Mapping

In [ ]:
#| export
def _domains_aa_to_genomic(exons, cds_bounds, domains_aa, strand):
    """
    Map domains defined in amino-acid coords (1-based, CDS-only)
    to genomic intervals, possibly split across exons.
    Strand-aware (fixes minus-strand mirroring/offset errors).
    """
    if cds_bounds is None:
        raise ValueError("No CDS bounds available for this transcript.")

    cs, ce = sorted(cds_bounds)

    # exon intervals in genomic coordinates (low->high)
    ex_sorted = [(min(a, b), max(a, b)) for a, b in exons]
    ex_sorted.sort(key=lambda x: x[0])

    # Build coding segments = exon ∩ CDS, in genomic order
    coding_segments = []
    cds_len = 0
    for s, e in ex_sorted:
        seg_start = max(s, cs)
        seg_end = min(e, ce)
        if seg_end > seg_start:
            seg_len = seg_end - seg_start
            coding_segments.append(
                {
                    "g_start": seg_start,
                    "g_end": seg_end,
                    "length": seg_len,
                    "cds_offset": None,  # assigned below
                }
            )
            cds_len += seg_len

    if cds_len <= 0:
        raise ValueError("CDS length is zero after intersecting with exons")

    # Normalize strand to a simple flag
    is_minus = strand in (-1, "-", "minus")

    # Assign CDS offsets in transcript CDS order
    # + strand: transcript order == increasing genomic
    # - strand: transcript order == decreasing genomic (reverse segments)
    segs_in_cds_order = list(reversed(coding_segments)) if is_minus else coding_segments

    off = 0
    for seg in segs_in_cds_order:
        seg["cds_offset"] = off
        off += seg["length"]

    out = {}
    for dom in domains_aa:
        dom_id = dom["name"]
        aa1, aa2 = int(dom["start"]), int(dom["end"])

        # AA coords are 1-based; convert to CDS nucleotide offsets (0-based half-open)
        nt_start = (aa1 - 1) * 3
        nt_end = aa2 * 3

        intervals = []
        for seg in segs_in_cds_order:
            seg_off = seg["cds_offset"]
            seg_len = seg["length"]

            local_start = max(0, nt_start - seg_off)
            local_end = min(seg_len, nt_end - seg_off)
            if local_end <= local_start:
                continue

            if is_minus:
                # Map from right edge of segment on minus strand
                g_s = seg["g_end"] - local_end
                g_e = seg["g_end"] - local_start
            else:
                # Map from left edge of segment on plus strand
                g_s = seg["g_start"] + local_start
                g_e = seg["g_start"] + local_end

            a, b = int(g_s), int(g_e)
            intervals.append((min(a, b), max(a, b)))

        out[dom_id] = intervals

    return out

## ProteinData Class

Main class for retrieving protein domain information and mapping to genomic coordinates.

## ProteinData Class

Main API for protein domain retrieval and coordinate mapping.

In [ ]:
#| export
class ProteinData:
    """Protein domain retrieval and genomic coordinate mapping.
    
    Provides methods for fetching protein domains from Ensembl or InterPro APIs
    and mapping amino acid coordinates to genomic positions.
    
    Parameters
    ----------
    transcript_data : TranscriptData
        TranscriptData instance for coordinate lookups
    provider : str, default "ensembl"
        Default domain provider ("ensembl" or "interpro")
    
    Examples
    --------
    >>> from allos.transcript_data import TranscriptData
    >>> td = TranscriptData(gtf_file="genes.gtf")
    >>> pd = ProteinData(td, provider="ensembl")
    >>> domains = pd.get_protein_domains("ENST00000317610", max_domains=5)
    """
    
    def __init__(self, transcript_data, provider: str = "ensembl"):
        """Initialize with TranscriptData instance."""
        self.transcript_data = transcript_data
        self.provider = provider
    
    def get_protein_id(self, transcript_id: str) -> Optional[str]:
        """Get Ensembl protein ID (ENSP) for transcript.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID (e.g., ENST00000317610)
        
        Returns
        -------
        protein_id : str or None
            Protein ID (e.g., ENSP00000...) or None if no translation
        """
        pid, _, _ = _fetch_translation_and_features(transcript_id, self.provider)
        return pid
    
    def get_protein_length(self, transcript_id: str) -> Optional[int]:
        """Get protein length in amino acids.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        
        Returns
        -------
        length : int or None
            Protein length in AA, or None if no translation
        """
        _, plen, _ = _fetch_translation_and_features(transcript_id, self.provider)
        return plen if plen > 0 else None
    
    def get_raw_features(
        self,
        transcript_id: str,
        provider: Optional[str] = None,
    ) -> List[Dict]:
        """Get raw feature data from API without filtering.
        
        Use this to inspect what features are available before filtering.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        provider : str, optional
            Domain provider ("ensembl" or "interpro")
            If None, uses instance default
        
        Returns
        -------
        features : List[Dict]
            Raw feature dicts with keys: start, end, id, desc, source
        """
        prov = provider if provider is not None else self.provider
        _, _, features = _fetch_translation_and_features(transcript_id, prov)
        return features
    
    def inspect_features(
        self,
        transcript_id: str,
        provider: Optional[str] = None,
    ) -> Dict:
        """Inspect available features and their metadata.
        
        Returns a summary of feature types, sources, and IDs for filtering.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        provider : str, optional
            Domain provider
        
        Returns
        -------
        summary : Dict
            Summary with keys:
            - 'feature_types': unique feature types
            - 'sources': unique source databases
            - 'id_prefixes': common ID prefixes (first 2-3 chars)
            - 'features': list of raw features
        """
        features = self.get_raw_features(transcript_id, provider)
        
        if not features:
            return {
                'feature_types': [],
                'sources': [],
                'id_prefixes': [],
                'features': []
            }
        
        # Extract unique values
        feature_types = sorted(set(f.get('source', 'unknown') for f in features))
        sources = sorted(set(f.get('source', 'unknown') for f in features))
        
        # Get common ID prefixes
        ids = [str(f.get('id', '')) for f in features]
        prefixes = set()
        for fid in ids:
            if len(fid) >= 2:
                prefixes.add(fid[:2])
            if len(fid) >= 3:
                prefixes.add(fid[:3])
        
        return {
            'feature_types': feature_types,
            'sources': sources,
            'id_prefixes': sorted(prefixes),
            'features': features
        }
    
    def get_protein_domains(
        self,
        transcript_id: str,
        provider: Optional[str] = None,
        max_domains: int = 12,
        id_prefixes: Optional[List[str]] = None,
        sources: Optional[List[str]] = None,
    ) -> List[Dict]:
        """Get protein domains in amino acid coordinates.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        provider : str, optional
            Domain provider ("ensembl" or "interpro")
            If None, uses instance default
        max_domains : int, default 12
            Maximum number of domains to return
        id_prefixes : List[str], optional
            Filter domains by ID prefix (e.g., ["PF"] for Pfam)
        sources : List[str], optional
            Filter by source database (e.g., ["pfam", "smart"])
        
        Returns
        -------
        domains : List[Dict]
            List of domain dicts with keys: name, label, start, end
            Coordinates are 1-based amino acid positions
        """
        prov = provider if provider is not None else self.provider
        _, _, features = _fetch_translation_and_features(transcript_id, prov)
        
        if not features:
            return []
        
        # Filter by sources if specified
        if sources is not None:
            features = [f for f in features if f.get('source') in sources]
        
        # Convert features to domains (with id_prefixes filtering)
        domains = _features_to_domains_aa(features, max_domains=max_domains, id_prefixes=id_prefixes)
        return domains
    
    def domains_aa_to_genomic(
        self,
        transcript_id: str,
        domains_aa: List[Dict],
    ) -> Dict[str, List[Tuple[int, int]]]:
        """Map amino acid domain coordinates to genomic intervals.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        domains_aa : List[Dict]
            Domain dicts with 'name', 'start', 'end' keys (AA coords)
        
        Returns
        -------
        domains_genomic : Dict[str, List[Tuple[int, int]]]
            Mapping from domain name to list of genomic intervals (start, end)
            Handles exon boundaries - a single domain may span multiple intervals
        """
        # Get exons and strand from TranscriptData
        exons, strand = self.transcript_data.get_exon_coords_and_strand(transcript_id)
        
        # Get CDS array and compute bounds
        cds = self.transcript_data._idx.cds(transcript_id)
        if cds.size == 0:
            return {}
        
        cds_bounds = (int(cds[:, 0].min()), int(cds[:, 1].max()))
        
        # Use helper function to map AA to genomic
        return _domains_aa_to_genomic(exons, cds_bounds, domains_aa, strand)

    def aa_to_genomic(
        self,
        transcript_id: str,
        aa_pos: int,
    ) -> Optional[int]:
        """Map a single amino acid position to genomic coordinate.

        Parameters
        ----------
        transcript_id : str
            Transcript ID
        aa_pos : int
            Amino acid position (1-based)

        Returns
        -------
        genomic_pos : int or None
            Genomic coordinate for the start of this codon, or None if mapping fails
        """
        # Get exons and strand from TranscriptData
        exons, strand = self.transcript_data.get_exon_coords_and_strand(transcript_id)

        # Get CDS array and compute bounds
        cds = self.transcript_data._idx.cds(transcript_id)
        if cds.size == 0:
            return None

        cds_bounds = (int(cds[:, 0].min()), int(cds[:, 1].max()))
        cs, ce = sorted(cds_bounds)

        # Build coding segments
        ex_sorted = [(min(a, b), max(a, b)) for a, b in exons]
        ex_sorted.sort(key=lambda x: x[0])

        coding_segments = []
        for s, e in ex_sorted:
            seg_start = max(s, cs)
            seg_end = min(e, ce)
            if seg_end > seg_start:
                coding_segments.append({
                    "g_start": seg_start,
                    "g_end": seg_end,
                    "length": seg_end - seg_start,
                    "cds_offset": None,
                })

        if not coding_segments:
            return None

        is_minus = strand in (-1, "-", "minus")
        segs_in_cds_order = list(reversed(coding_segments)) if is_minus else coding_segments

        # Assign CDS offsets
        off = 0
        for seg in segs_in_cds_order:
            seg["cds_offset"] = off
            off += seg["length"]

        # Convert AA position to nucleotide offset (0-based)
        nt_offset = (aa_pos - 1) * 3

        # Find which segment contains this offset
        for seg in segs_in_cds_order:
            seg_off = seg["cds_offset"]
            seg_len = seg["length"]

            if seg_off <= nt_offset < seg_off + seg_len:
                local_offset = nt_offset - seg_off
                if is_minus:
                    return seg["g_end"] - local_offset - 1
                else:
                    return seg["g_start"] + local_offset

        return None
    
    def get_protein_domains_genomic(
        self,
        transcript_id: str,
        **kwargs
    ) -> Tuple[List[Dict], Dict[str, List[Tuple[int, int]]]]:
        """Convenience method: get domains in both AA and genomic coords.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        **kwargs
            Passed to get_protein_domains()
        
        Returns
        -------
        domains_aa : List[Dict]
            Domains in AA coordinates
        domains_genomic : Dict[str, List[Tuple]]
            Same domains in genomic coordinates
        """
        domains_aa = self.get_protein_domains(transcript_id, **kwargs)
        domains_genomic = self.domains_aa_to_genomic(transcript_id, domains_aa)
        return domains_aa, domains_genomic

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()